## Prediction Pipeline Setup

**Model Loading Behavior:**
- MLflow caches model artifacts locally by version
- If 'production' alias points to a NEW version → loads fresh model
- If 'production' alias points to SAME version → uses cached artifacts
- Re-running cell 1 checks the current alias and loads accordingly

In [27]:
import sys
from pathlib import Path

# Add parent directory to path to import data_pipeline modules
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from data_pipeline.database import save_to_db, initialize_tables, update_pipeline_run_status
from infra.db.database_utils import DatabaseFactory
import os

import logging
logger = logging.getLogger(__name__)
import mlflow
from load_model import load_production_model

# Set up MLflow with SQLite backend for model registry
mlflow_db = project_root / 'model_training' / 'mlflow.db'
mlflow_artifacts = project_root / 'model_training' / 'mlruns'

# Format: sqlite:///path/to/db with artifact location
MLFLOW_TRACKING_URI = f"sqlite:///{mlflow_db.as_posix()}"

MODEL_NAME = os.getenv("MODEL_NAME", "HAR_xgboost")
MODEL_STAGE = os.getenv("MODEL_STAGE", "production") 



database_engine = DatabaseFactory.create_engine(
    db_type='sqlite',
    db_path=str(project_root / 'sensor_features.db')  # Use absolute path from project root
)


# load model - REFRESH THE MODEL AFTER RETRAINING
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")

# Clear any cached models
import importlib
import load_model
importlib.reload(load_model)
from load_model import load_production_model

model = load_production_model(MODEL_NAME, MODEL_STAGE)
print(f"✓ Model loaded successfully")

# Check model signature
print(f"\nModel input schema:")
if hasattr(model, 'metadata') and model.metadata.signature:
    input_schema = model.metadata.signature.inputs
    print(f"  Expected features: {len(input_schema.input_names())}")
    print(f"  First 5 features: {input_schema.input_names()[:5]}")
else:
    print("  No signature found in model metadata")



# run prediction on the whole dataset and apply labels

MLflow tracking URI: sqlite:///E:/src/neat-calculator/model_training/mlflow.db
Loading model from MLflow URI: models:/HAR_xgboost@production
✓ Model loaded successfully

Model input schema:
  No signature found in model metadata


In [28]:
# Load data to predict
data_to_predict = database_engine.get_records(table_name="training_data_labeled")

print(f"Found {len(data_to_predict)} rows to predict")
print(f"Columns: {data_to_predict.columns.tolist()[:10]}...")  # Show first 10 columns

# Drop non-feature columns
columns_to_drop = ["Activity", "timestamp"]
feature_columns = [col for col in data_to_predict.columns if col not in columns_to_drop]

X = data_to_predict[feature_columns].to_numpy()
print(f"Feature matrix shape: {X.shape}")

# Make predictions with probabilities
predictions = model.predict(data_to_predict[feature_columns])  # class labels
probabilities = model.predict_proba(data_to_predict[feature_columns])  # probability matrix

print(f"\nPredictions shape: {predictions.shape}")
print(f"Probabilities shape: {probabilities.shape}")
print(f"\nFirst 5 predictions: {predictions[:5]}")
print(f"First prediction probabilities: {probabilities[0]}")

Found 631 rows to predict
Columns: ['tBodyAcc-mean()-X', 'tBodyAcc-std()-X', 'tBodyAcc-min()-X', 'tBodyAcc-max()-X', 'tBodyAcc-energy()-X', 'tBodyAcc-entropy()-X', 'tBodyAcc-mean()-Y', 'tBodyAcc-std()-Y', 'tBodyAcc-min()-Y', 'tBodyAcc-max()-Y']...
Feature matrix shape: (631, 180)

Predictions shape: (631,)
Probabilities shape: (631, 5)

First 5 predictions: [2 2 2 2 2]
First prediction probabilities: [1.0468749e-03 1.1388087e-03 9.9652928e-01 4.9164362e-04 7.9343922e-04]


In [29]:
# Interpret predictions with probabilities
import pandas as pd
import numpy as np

# Get class labels from the model
class_labels = model.classes_

print(f"Activity classes (encoded): {class_labels}")
print(f"\nEach prediction has {len(class_labels)} probabilities, one for each activity class")

# Create a DataFrame with predictions and probabilities
results_df = pd.DataFrame({
    'timestamp': data_to_predict['timestamp'],
    'predicted_class': predictions,
    'confidence': np.max(probabilities, axis=1),  # Highest probability
})

# Add probability columns for each class
for idx, class_label in enumerate(class_labels):
    results_df[f'prob_class_{class_label}'] = probabilities[:, idx]

print(f"\n=== Prediction Summary ===")
print(f"Total predictions: {len(predictions)}")
print(f"\nPredictions by class:")
for class_id in class_labels:
    count = np.sum(predictions == class_id)
    pct = count / len(predictions) * 100
    print(f"  Class {class_id}: {count} predictions ({pct:.1f}%)")

print(f"\n=== Confidence Analysis ===")
print(f"Average confidence: {results_df['confidence'].mean():.2%}")
print(f"Min confidence: {results_df['confidence'].min():.2%}")
print(f"Max confidence: {results_df['confidence'].max():.2%}")

# Show high confidence predictions
print(f"\n=== High Confidence Predictions (>95%) ===")
high_conf = results_df[results_df['confidence'] > 0.95].head(5)
print(high_conf[['predicted_class', 'confidence'] + [col for col in results_df.columns if col.startswith('prob_')]])

# Show low confidence predictions
print(f"\n=== Low Confidence Predictions (<80%) ===")
low_conf = results_df[results_df['confidence'] < 0.80].head(5)
if len(low_conf) > 0:
    print(low_conf[['predicted_class', 'confidence'] + [col for col in results_df.columns if col.startswith('prob_')]])
else:
    print("No low confidence predictions found!")

# Show first 3 predictions in detail
print(f"\n=== First 3 Predictions in Detail ===")
for i in range(min(3, len(predictions))):
    print(f"\nPrediction {i+1}:")
    print(f"  Predicted class: {predictions[i]}")
    print(f"  Confidence: {np.max(probabilities[i]):.2%}")
    print(f"  All probabilities:")
    for class_id, prob in zip(class_labels, probabilities[i]):
        bar = '█' * int(prob * 50)
        print(f"    Class {class_id}: {prob:.4f} ({prob*100:.2f}%) {bar}")

Activity classes (encoded): [0 1 2 3 4]

Each prediction has 5 probabilities, one for each activity class

=== Prediction Summary ===
Total predictions: 631

Predictions by class:
  Class 0: 13 predictions (2.1%)
  Class 1: 25 predictions (4.0%)
  Class 2: 429 predictions (68.0%)
  Class 3: 132 predictions (20.9%)
  Class 4: 32 predictions (5.1%)

=== Confidence Analysis ===
Average confidence: 98.13%
Min confidence: 44.63%
Max confidence: 99.99%

=== High Confidence Predictions (>95%) ===
   predicted_class  confidence  prob_class_0  prob_class_1  prob_class_2  \
0                2    0.996529      0.001047      0.001139      0.996529   
2                2    0.991918      0.006794      0.000787      0.991918   
3                2    0.999312      0.000134      0.000302      0.999312   
4                2    0.994628      0.000559      0.000868      0.994628   
5                2    0.998353      0.000328      0.000693      0.998353   

   prob_class_3  prob_class_4  
0      0.000492 

## Understanding the Probabilities

The model outputs probabilities for each activity class:
- **Probabilities sum to 1.0** (100%) for each prediction
- **Higher probability = more confident** the model is about that class
- **Confidence** = the highest probability among all classes
- **Good predictions** typically have >90% confidence in one class
- **Uncertain predictions** have probabilities spread across multiple classes